# Five-threshold directional change: N-scale hierarchy and grid design

This notebook uses five thresholds as a worked example of a general N-threshold hierarchy. Five is not treated as a canonical market calibration.

Two grids are fixed before inspecting their results:

- linear: 0.20%, 0.40%, 0.60%, 0.80%, 1.00%;
- constant ratio: 0.20%, 0.30%, 0.45%, 0.675%, 1.0125%.

Both grids use the same GBPUSD.r close series and the same causal directional-change definitions. The comparison asks what threshold geometry changes:

1. event-clock frequency and completed-event TMV, T, and R;
2. the occupancy of the 32 possible live-mode configurations;
3. coherent propagation depth, agreement breadth, and fragmentation;
4. adjacent-scale propagation and nesting;
5. the causal structural reading of the same price path.

No grid is selected using later returns or trading performance. Adjacent propagated/rejected outcomes and the selected chart interval are explicitly diagnostic hindsight.

In [1]:
from enum import StrEnum
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from dc_basics.core import dc_live_features, parse_dc_events

from dc_basics.data import resolve_data_root

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
N_PRICES = 20_000
DIVERGENCE_BARS_BEFORE = 180
DIVERGENCE_BARS_AFTER = 180

In [2]:
class ThresholdGrid(StrEnum):
    LINEAR = "linear"
    CONSTANT_RATIO = "constant_ratio"


class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"
    OUTER = "outer"


from dc_basics.enums import DcMode


class HierarchyShape(StrEnum):
    BROAD_AGREEMENT = "broad_agreement"
    COUNTER_DEPTH_1 = "counter_depth_1"
    COUNTER_DEPTH_2 = "counter_depth_2"
    COUNTER_DEPTH_3 = "counter_depth_3"
    COUNTER_DEPTH_4 = "counter_depth_4"
    FRAGMENTED_2_BOUNDARIES = "fragmented_2_boundaries"
    FRAGMENTED_3_BOUNDARIES = "fragmented_3_boundaries"
    FRAGMENTED_4_BOUNDARIES = "fragmented_4_boundaries"


from dc_basics.enums import AttemptOutcome


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO, Scale.OUTER)
GRID_THRESHOLDS = {
    ThresholdGrid.LINEAR: {
        Scale.SMALL: 0.002,
        Scale.MEDIUM: 0.004,
        Scale.LARGE: 0.006,
        Scale.MACRO: 0.008,
        Scale.OUTER: 0.010,
    },
    ThresholdGrid.CONSTANT_RATIO: {
        Scale.SMALL: 0.002,
        Scale.MEDIUM: 0.003,
        Scale.LARGE: 0.0045,
        Scale.MACRO: 0.00675,
        Scale.OUTER: 0.010125,
    },
}
GRID_LABELS = {
    ThresholdGrid.LINEAR: "Linear absolute spacing",
    ThresholdGrid.CONSTANT_RATIO: "Constant ratio 1.5",
}
GRID_PREFIXES = {
    ThresholdGrid.LINEAR: "LIN",
    ThresholdGrid.CONSTANT_RATIO: "GEO",
}
SCALE_VISUALS = {
    Scale.SMALL: {
        "prefix": "S", "up_color": "#16a34a", "down_color": "#dc2626",
        "line_width": 0.7, "line_dash": "solid",
    },
    Scale.MEDIUM: {
        "prefix": "M", "up_color": "#2563eb", "down_color": "#f59e0b",
        "line_width": 1.0, "line_dash": "dash",
    },
    Scale.LARGE: {
        "prefix": "L", "up_color": "#7c3aed", "down_color": "#0f766e",
        "line_width": 1.3, "line_dash": "dot",
    },
    Scale.MACRO: {
        "prefix": "X", "up_color": "#c026d3", "down_color": "#92400e",
        "line_width": 1.7, "line_dash": "longdash",
    },
    Scale.OUTER: {
        "prefix": "O", "up_color": "#0891b2", "down_color": "#be123c",
        "line_width": 2.1, "line_dash": "dashdot",
    },
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

for grid, thresholds in GRID_THRESHOLDS.items():
    ordered = [thresholds[scale] for scale in SCALE_ORDER]
    if not all(left < right for left, right in zip(ordered[:-1], ordered[1:], strict=True)):
        raise ValueError(f"{grid.value} thresholds must be strictly increasing.")

threshold_contract = pd.DataFrame({
    grid.value: [GRID_THRESHOLDS[grid][scale] for scale in SCALE_ORDER]
    for grid in ThresholdGrid
}, index=[scale.value for scale in SCALE_ORDER])
threshold_contract.index.name = "ordinal_scale"
threshold_contract.style.format("{:.4%}")

,linear,constant_ratio
ordinal_scale,,
small,0.2000%,0.2000%
medium,0.4000%,0.3000%
large,0.6000%,0.4500%
macro,0.8000%,0.6750%
outer,1.0000%,1.0125%


## Five-scale reading and causality contract

Each grid creates five independent event clocks from one price path. A bar configuration is read in this fixed order:

    small / medium / large / macro / outer

For example:

    down / down / up / up / up

has one boundary and is a coherent counter-move that has propagated through two scales. By contrast:

    down / up / down / up / up

has three boundaries and is fragmented.

All five live modes and their derived structural properties are causal once all clocks have initialized. Completed-event TMV, T, and R become known only at their DCC confirmation. A propagated/rejected label becomes known only on its resolution DCC.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")

prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Symbol: {SYMBOL}")
print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Symbol: GBPUSD.r
Bars: 259,094
Range: 2023-05-22 14:00:00+02:00 to 2026-11-11 08:55:00+02:00


## Build both five-clock systems

The event identifier contains the grid and scale so that no completed event can be confused across calibrations. The first threshold is identical in both grids; its event clock must therefore be identical and is used as an implementation invariant.

In [4]:
from dc_basics.events import prepare_event_ledger as _prepare_event_ledger

def prepare_event_ledger(source: pd.DataFrame, grid: ThresholdGrid, scale: Scale, theta: float) -> pd.DataFrame:
    event_prefix = f"{GRID_PREFIXES[grid]}-{SCALE_VISUALS[scale]['prefix']}"
    ledger = _prepare_event_ledger(source, event_prefix, scale, theta)
    ledger.insert(1, "grid", grid.value)
    return ledger


grid_artifacts = {}
for grid in ThresholdGrid:
    event_ledgers = {}
    live_frames = {}
    mode_frame = pd.DataFrame(index=prices.index)

    for scale in SCALE_ORDER:
        theta = GRID_THRESHOLDS[grid][scale]
        raw_events = parse_dc_events(prices=prices, theta=theta)
        ledger = prepare_event_ledger(raw_events, grid, scale, theta)
        live = dc_live_features(prices, theta=theta)
        event_ledgers[scale] = ledger
        live_frames[scale] = live
        mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
        mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
        mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
        event_lookup = ledger.set_index("dcc_idx")["event_id"]
        mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(event_lookup)

    grid_artifacts[grid] = {
        "event_ledgers": event_ledgers,
        "live_frames": live_frames,
        "mode_frame": mode_frame,
    }

linear_small_times = grid_artifacts[ThresholdGrid.LINEAR]["event_ledgers"][Scale.SMALL]["dcc_idx"]
ratio_small_times = grid_artifacts[ThresholdGrid.CONSTANT_RATIO]["event_ledgers"][Scale.SMALL]["dcc_idx"]
pd.testing.assert_series_equal(
    linear_small_times.reset_index(drop=True),
    ratio_small_times.reset_index(drop=True),
    check_names=False,
)
print("Shared 0.20% clock invariant: passed")

Shared 0.20% clock invariant: passed


## Clock-level comparison

TMV, T, and R below belong to completed prior legs and are known at each event's DCC confirmation. Equal absolute spacing and equal proportional spacing ask different questions. The comparison is descriptive; it does not declare a winning grid.

In [5]:
clock_records = []
for grid in ThresholdGrid:
    previous_theta = np.nan
    for scale in SCALE_ORDER:
        ledger = grid_artifacts[grid]["event_ledgers"][scale]
        theta = GRID_THRESHOLDS[grid][scale]
        dcc_spacing = ledger["dcc_pos"].diff()
        clock_records.append({
            "grid": grid.value,
            "scale": scale.value,
            "threshold": theta,
            "threshold_ratio_to_previous": theta / previous_theta if np.isfinite(previous_theta) else np.nan,
            "events": len(ledger),
            "median_t": ledger["t"].median(),
            "mean_t": ledger["t"].mean(),
            "median_tmv": ledger["tmv"].median(),
            "median_r": ledger["r"].median(),
            "median_bars_between_dccs": dcc_spacing.median(),
        })
        previous_theta = theta

clock_summary = pd.DataFrame.from_records(clock_records).set_index(["grid", "scale"])
display(clock_summary.style.format({
    "threshold": "{:.4%}",
    "threshold_ratio_to_previous": "{:.3f}",
    "events": "{:,}",
    "median_t": "{:.1f}",
    "mean_t": "{:.1f}",
    "median_tmv": "{:.3f}",
    "median_r": "{:.3e}",
    "median_bars_between_dccs": "{:.1f}",
}))

clock_figure = make_subplots(rows=1, cols=2, subplot_titles=["Completed events", "Median completed-leg T"])
for grid in ThresholdGrid:
    summary = clock_summary.loc[grid.value]
    clock_figure.add_trace(go.Bar(
        x=summary.index,
        y=summary["events"],
        name=GRID_LABELS[grid],
        legendgroup=grid.value,
        marker_color="#2563eb" if grid == ThresholdGrid.LINEAR else "#f97316",
    ), row=1, col=1)
    clock_figure.add_trace(go.Scatter(
        x=summary.index,
        y=summary["median_t"],
        mode="lines+markers",
        name=GRID_LABELS[grid],
        legendgroup=grid.value,
        showlegend=False,
        line={"color": "#2563eb" if grid == ThresholdGrid.LINEAR else "#f97316", "width": 2},
    ), row=1, col=2)

clock_figure.update_layout(
    title=f"{SYMBOL}: five-clock event frequency and duration by grid",
    template="plotly_white",
    height=500,
    barmode="group",
    legend={"orientation": "h", "y": 1.15, "x": 0.0},
)
clock_figure.update_yaxes(title="Events", row=1, col=1)
clock_figure.update_yaxes(title="Bars", row=1, col=2)
clock_figure.show()

## Completed-event overlays

Each figure shows one grid at a time. A connected extreme-to-extreme segment is a completed prior leg. A DCC triangle is the confirmation that ends that leg and establishes the new live mode. The five scales use distinct line styles and colour pairs.

In [6]:
def direction_color(scale: Scale, direction: str) -> str:
    key = "up_color" if direction == DcMode.UP.value else "down_color"
    return SCALE_VISUALS[scale][key]


def plot_threshold_events(
    close: pd.Series, ledgers: dict[Scale, pd.DataFrame], grid: ThresholdGrid, n_prices: int
) -> go.Figure:
    if n_prices <= 0:
        raise ValueError("n_prices must be positive.")
    price_window = close.tail(n_prices)
    window_start = price_window.index[0]
    window_end = price_window.index[-1]
    figure = go.Figure()
    figure.add_trace(go.Scattergl(
        x=price_window.index,
        y=price_window.to_numpy(),
        mode="lines",
        name="Close",
        line={"color": "#475569", "width": 1.0},
        hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
    ))

    for scale in SCALE_ORDER:
        visual = SCALE_VISUALS[scale]
        theta = GRID_THRESHOLDS[grid][scale]
        events = ledgers[scale]
        visible = events.loc[events["ext_start_idx"].le(window_end) & events["dcc_idx"].ge(window_start)]

        for completed_direction in DcMode:
            directional = visible.loc[visible["direction"].eq(completed_direction.value)]
            x_values = []
            y_values = []
            for event in directional.itertuples(index=False):
                x_values.extend([event.ext_start_idx, event.ext_end_idx, None])
                y_values.extend([event.ext_start_price, event.ext_end_price, None])

            if x_values:
                figure.add_trace(go.Scatter(
                    x=x_values,
                    y=y_values,
                    mode="lines",
                    name=f"{scale.value.title()} {theta:.4%} completed {completed_direction.value}",
                    legendgroup=f"{grid.value}_{scale.value}_{completed_direction.value}",
                    line={
                        "color": direction_color(scale, completed_direction.value),
                        "width": visual["line_width"],
                        "dash": visual["line_dash"],
                    },
                    opacity=0.78,
                    hoverinfo="skip",
                ))

            confirmations = directional.loc[directional["dcc_idx"].between(window_start, window_end)]
            if confirmations.empty:
                continue
            live_direction = DcMode.DOWN.value if completed_direction == DcMode.UP else DcMode.UP.value
            symbol = "triangle-down" if live_direction == DcMode.DOWN.value else "triangle-up"
            figure.add_trace(go.Scatter(
                x=confirmations["dcc_idx"],
                y=confirmations["dcc_price"],
                mode="markers",
                name=f"{scale.value.title()} {live_direction} DCC",
                legendgroup=f"{grid.value}_{scale.value}_{live_direction}",
                showlegend=False,
                marker={
                    "color": direction_color(scale, live_direction),
                    "size": 4 + SCALE_ORDER.index(scale),
                    "symbol": symbol,
                    "line": {"color": "white", "width": 0.5},
                },
                customdata=confirmations[["event_id", "tmv", "t", "r", "information_time"]].to_numpy(),
                hovertemplate=(
                    f"{scale.value.title()} {live_direction} DCC<br>"
                    "time=%{x}<br>price=%{y:.5f}<br>event=%{customdata[0]}<br>"
                    "TMV=%{customdata[1]:.3f}<br>T=%{customdata[2]}<br>"
                    "R=%{customdata[3]:.3e}<br>known at=%{customdata[4]}<extra></extra>"
                ),
            ))

    figure.update_layout(
        title=f"{SYMBOL}: {GRID_LABELS[grid]} — five independent DC clocks, last {len(price_window):,} prices",
        template="plotly_white",
        height=820,
        hovermode="closest",
        legend={"orientation": "h", "y": 1.14, "x": 0.0},
        margin={"l": 65, "r": 30, "t": 155, "b": 65},
    )
    figure.update_xaxes(title="Time", range=[window_start, window_end])
    figure.update_yaxes(title="Close")
    return figure


for selected_grid in ThresholdGrid:
    plot_threshold_events(
        prices, grid_artifacts[selected_grid]["event_ledgers"], selected_grid, N_PRICES
    ).show()

## The 32 causal configurations

Five binary modes produce 32 possible configurations. The classification is direction-neutral:

- zero boundaries: broad agreement;
- one boundary: a coherent counter-front with depth 1 through 4;
- two or more boundaries: fragmentation.

In the code, `coherent_frontier` means **at most one** boundary, so it includes broad agreement (a frontier of depth 0); `one_boundary_share` isolates the single-boundary configurations and `fragmented_share` is the complement of `coherent_frontier_share`.

Outer agreement breadth counts how many of the five clocks currently agree with the outer mode. It is not sufficient by itself to identify shape because ordering matters.

In [7]:
from dc_basics.configuration import classify_configuration as _classify_configuration

def classify_configuration(modes: tuple[str, ...]) -> dict[str, object]:
    return _classify_configuration(modes, outer_scale_name="outer", expected_count=5)


configuration_reference = pd.DataFrame.from_records([
    classify_configuration(tuple(modes))
    for modes in product([DcMode.UP.value, DcMode.DOWN.value], repeat=len(SCALE_ORDER))
]).set_index("configuration")

if len(configuration_reference) != 32:
    raise AssertionError("The five-clock configuration reference must contain 32 states.")

configuration_reference

,hierarchy_shape,direction_boundaries,outer_agreement_breadth,small_coherence_depth,counter_depth,coherent_frontier
configuration,,,,,,
up/up/up/up/up,broad_agreement,0,5,5,0,True
up/up/up/up/down,counter_depth_4,1,1,4,4,True
up/up/up/down/up,fragmented_2_boundaries,2,4,3,<NA>,False
up/up/up/down/down,counter_depth_3,1,2,3,3,True
up/up/down/up/up,fragmented_2_boundaries,2,4,2,<NA>,False
up/up/down/up/down,fragmented_3_boundaries,3,2,2,<NA>,False
up/up/down/down/up,fragmented_2_boundaries,2,3,2,<NA>,False
up/up/down/down/down,counter_depth_2,1,3,2,2,True
up/down/up/up/up,fragmented_2_boundaries,2,4,1,<NA>,False


In [8]:
def build_state_path(mode_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
    all_modes_known = mode_frame[mode_columns].notna().all(axis=1)
    known_modes = mode_frame.loc[all_modes_known, mode_columns]
    properties = known_modes.apply(
        lambda row: pd.Series(classify_configuration(tuple(row.to_numpy()))),
        axis=1,
    )
    state_path = pd.concat([known_modes, properties], axis=1).reset_index(names="time")
    state_path["bar_position"] = prices.index.get_indexer(state_path["time"])
    state_path["run_id"] = (
        state_path["configuration"].ne(state_path["configuration"].shift()).fillna(True).cumsum()
    )
    state_runs = state_path.groupby("run_id", sort=True).agg(
        configuration=("configuration", "first"),
        hierarchy_shape=("hierarchy_shape", "first"),
        direction_boundaries=("direction_boundaries", "first"),
        outer_agreement_breadth=("outer_agreement_breadth", "first"),
        counter_depth=("counter_depth", "first"),
        coherent_frontier=("coherent_frontier", "first"),
        start_time=("time", "first"),
        end_time=("time", "last"),
        start_position=("bar_position", "first"),
        end_position=("bar_position", "last"),
        bars=("configuration", "size"),
    ).reset_index(drop=True)
    return state_path, state_runs


state_paths = {}
state_runs_by_grid = {}
for grid in ThresholdGrid:
    state_path, state_runs = build_state_path(grid_artifacts[grid]["mode_frame"])
    state_paths[grid] = state_path
    state_runs_by_grid[grid] = state_runs

configuration_shares = pd.concat({
    grid.value: state_paths[grid]["configuration"].value_counts(normalize=True)
    for grid in ThresholdGrid
}, axis=1).fillna(0.0)
configuration_shares.index.name = "configuration"

shape_shares = pd.concat({
    grid.value: state_paths[grid]["hierarchy_shape"].value_counts(normalize=True)
    for grid in ThresholdGrid
}, axis=1).fillna(0.0)
shape_shares.index.name = "hierarchy_shape"

structural_records = []
for grid in ThresholdGrid:
    state_path = state_paths[grid]
    probabilities = state_path["configuration"].value_counts(normalize=True)
    entropy_bits = -(probabilities * np.log2(probabilities)).sum()
    structural_records.append({
        "grid": grid.value,
        "initialized_bars": len(state_path),
        "observed_configurations": state_path["configuration"].nunique(),
        "configuration_entropy_bits": entropy_bits,
        "effective_configurations": 2.0 ** entropy_bits,
        "broad_agreement_share": state_path["hierarchy_shape"].eq(
            HierarchyShape.BROAD_AGREEMENT.value
        ).mean(),
        "one_boundary_share": state_path["direction_boundaries"].eq(1).mean(),
        # coherent_frontier = at most one boundary, so it includes broad agreement (zero boundaries).
        "coherent_frontier_share": state_path["coherent_frontier"].mean(),
        "fragmented_share": (~state_path["coherent_frontier"]).mean(),
        "mean_direction_boundaries": state_path["direction_boundaries"].mean(),
        "mean_outer_agreement_breadth": state_path["outer_agreement_breadth"].mean(),
    })

structural_summary = pd.DataFrame.from_records(structural_records).set_index("grid")
display(structural_summary.style.format({
    "initialized_bars": "{:,}",
    "observed_configurations": "{:,}",
    "configuration_entropy_bits": "{:.3f}",
    "effective_configurations": "{:.2f}",
    "broad_agreement_share": "{:.2%}",
    "one_boundary_share": "{:.2%}",
    "coherent_frontier_share": "{:.2%}",
    "fragmented_share": "{:.2%}",
    "mean_direction_boundaries": "{:.3f}",
    "mean_outer_agreement_breadth": "{:.3f}",
}))
display(shape_shares.style.format("{:.2%}"))
display(configuration_shares.sort_index().style.format("{:.2%}"))

,initialized_bars,observed_configurations,configuration_entropy_bits,effective_configurations,broad_agreement_share,one_boundary_share,coherent_frontier_share,fragmented_share,mean_direction_boundaries,mean_outer_agreement_breadth
grid,,,,,,,,,,
linear,"258,340",32,4.160,17.88,28.42%,45.12%,73.54%,26.46%,1.029,3.633
constant_ratio,"258,334",32,4.372,20.71,26.35%,44.31%,70.66%,29.34%,1.095,3.367


,linear,constant_ratio
hierarchy_shape,,
broad_agreement,28.42%,26.35%
fragmented_2_boundaries,21.90%,23.43%
counter_depth_1,19.10%,11.43%
counter_depth_2,12.77%,11.15%
counter_depth_3,7.85%,11.55%
counter_depth_4,5.39%,10.19%
fragmented_3_boundaries,4.26%,5.29%
fragmented_4_boundaries,0.30%,0.62%


,linear,constant_ratio
configuration,,
down/down/down/down/down,12.42%,11.48%
down/down/down/down/up,2.62%,4.90%
down/down/down/up/down,0.38%,1.31%
down/down/down/up/up,3.78%,5.57%
down/down/up/down/down,1.30%,1.66%
down/down/up/down/up,0.46%,0.95%
down/down/up/up/down,0.88%,1.96%
down/down/up/up/up,6.39%,6.16%
down/up/down/down/down,3.06%,1.34%


## Live-mode ribbons

These ribbons contain only information available at each bar. They show how grid geometry changes the structural description even though the underlying close series is identical.

In [9]:
def plot_mode_ribbons(
    close: pd.Series, frame: pd.DataFrame, grid: ThresholdGrid, n_prices: int
) -> go.Figure:
    price_window = close.tail(n_prices)
    modes = frame.loc[price_window.index]
    figure = make_subplots(
        rows=6,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.64, 0.072, 0.072, 0.072, 0.072, 0.072],
        vertical_spacing=0.018,
        subplot_titles=["Close", *[scale.value.title() for scale in SCALE_ORDER]],
    )
    figure.add_trace(go.Scattergl(
        x=price_window.index,
        y=price_window.to_numpy(),
        mode="lines",
        name="Close",
        line={"color": "#475569", "width": 1.0},
        hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
    ), row=1, col=1)

    for row_number, scale in enumerate(SCALE_ORDER, start=2):
        theta = GRID_THRESHOLDS[grid][scale]
        for direction in DcMode:
            ribbon = np.where(modes[f"{scale.value}_mode"].eq(direction.value), 1.0, np.nan)
            figure.add_trace(go.Scattergl(
                x=modes.index,
                y=ribbon,
                mode="lines",
                name=f"{scale.value.title()} {theta:.4%} {direction.value}",
                legendgroup=f"{grid.value}_{scale.value}_{direction.value}_mode",
                line={"color": direction_color(scale, direction.value), "width": 11},
                connectgaps=False,
                hovertemplate=(
                    f"{scale.value.title()} mode={direction.value}<br>"
                    f"theta={theta:.4%}<br>time=%{{x}}<extra></extra>"
                ),
            ), row=row_number, col=1)
        figure.update_yaxes(showticklabels=False, range=[0.9, 1.1], row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: {GRID_LABELS[grid]} causal five-scale modes — last {len(price_window):,} prices",
        template="plotly_white",
        height=980,
        hovermode="x unified",
        legend={"orientation": "h", "y": 1.10, "x": 0.0},
    )
    figure.update_xaxes(title="Time", row=6, col=1)
    return figure


for selected_grid in ThresholdGrid:
    plot_mode_ribbons(prices, grid_artifacts[selected_grid]["mode_frame"], selected_grid, N_PRICES).show()

## Configuration-transition integrity

A configuration component may change only on its own DCC. More than one component can change on the same close bar, but close-only data cannot recover the intrabar ordering of those confirmations.

In [10]:
transition_records = []
transition_signatures = {}
for grid in ThresholdGrid:
    state_path = state_paths[grid]
    mode_frame = grid_artifacts[grid]["mode_frame"]
    previous_configuration = state_path["configuration"].shift()
    is_transition = state_path["configuration"].ne(previous_configuration) & previous_configuration.notna()
    transitions = state_path.loc[is_transition, ["time", "configuration", "hierarchy_shape"]].copy()
    signatures = []

    for transition in transitions.itertuples(index=False):
        time = transition.time
        position = state_path.index[state_path["time"].eq(time)][0]
        changed_scales = []
        for scale in SCALE_ORDER:
            mode_column = f"{scale.value}_mode"
            current_mode = state_path.at[position, mode_column]
            previous_mode = state_path.at[position - 1, mode_column]
            changed = current_mode != previous_mode
            dcc_columns = [f"{scale.value}_upward_dcc", f"{scale.value}_downward_dcc"]
            observed_dcc = bool(mode_frame.loc[time, dcc_columns].any())
            if changed != observed_dcc:
                raise AssertionError(f"{grid.value} {scale.value} transition disagrees with its DCC flags.")
            if changed:
                changed_scales.append(scale.value)
        signatures.append("+".join(changed_scales))

    transitions["transition_signature"] = signatures
    transitions["changed_scale_count"] = transitions["transition_signature"].str.count(r"\+") + 1
    transition_signatures[grid] = transitions
    transition_records.append({
        "grid": grid.value,
        "configuration_transitions": len(transitions),
        "multi_scale_same_bar_transitions": transitions["changed_scale_count"].gt(1).sum(),
        "multi_scale_same_bar_share": transitions["changed_scale_count"].gt(1).mean(),
        "observed_transition_signatures": transitions["transition_signature"].nunique(),
    })

transition_summary = pd.DataFrame.from_records(transition_records).set_index("grid")
display(transition_summary.style.format({
    "configuration_transitions": "{:,}",
    "multi_scale_same_bar_transitions": "{:,}",
    "multi_scale_same_bar_share": "{:.2%}",
    "observed_transition_signatures": "{:,}",
}))

signature_counts = pd.concat({
    grid.value: transition_signatures[grid]["transition_signature"].value_counts()
    for grid in ThresholdGrid
}, axis=1).fillna(0).astype(int)
signature_counts.index.name = "changed_scales"
display(signature_counts)

,configuration_transitions,multi_scale_same_bar_transitions,multi_scale_same_bar_share,observed_transition_signatures
grid,,,,
linear,"4,205",73,1.74%,18
constant_ratio,"5,068",116,2.29%,18


,linear,constant_ratio
changed_scales,,
small,2632,2600
medium,786,1293
large,367,631
macro,210,289
outer,137,139
small+medium,36,51
small+large,7,9
small+medium+large,6,18
medium+large,6,15


## Four adjacent propagation contests

Each edge is the familiar two-clock contest. A lower-scale counter DCC initiates an attempt. It propagates if the upper DCC confirms the same target mode first; it is rejected if the lower clock flips back first.

The initiating modes are causal. The final propagated/rejected outcome is hindsight until the resolution bar. Same-bar resolutions are retained separately because their intrabar order is unknowable from close-only bars.

**Why `approximate_propagation_rate` works.** Between two consecutive upper DCCs the lower clock fires an odd number $n$ of times (nesting parity, below). Those firings alternate seed, rejection, …, seed, and the last seed propagates, so each upper interval holds $(n+1)/2$ attempts and exactly one propagation. Summed over intervals that is about $(N_{lower}+N_{upper})/2$ attempts and $N_{upper}$ propagations, so the propagation rate is about $2/(N_{lower}/N_{upper}+1)$. Compare it with `all_resolved_propagation_rate`, which counts same-bar propagations exactly as the arithmetic does; `clean_propagation_rate` drops them and is therefore slightly lower. When the approximation matches, an adjacent propagation rate is mostly a consequence of the two event counts — and event-count ratios follow threshold ratios — rather than a separate behavioural fact about the market.

In [11]:
from dc_basics.hierarchy import build_adjacent_attempts


adjacent_pairs = tuple(zip(SCALE_ORDER[:-1], SCALE_ORDER[1:], strict=True))
attempts_by_grid = {}
edge_records = []

for grid in ThresholdGrid:
    mode_frame = grid_artifacts[grid]["mode_frame"]
    ledgers = grid_artifacts[grid]["event_ledgers"]
    attempts = pd.concat([
        build_adjacent_attempts(mode_frame, lower_scale, upper_scale)
        for lower_scale, upper_scale in adjacent_pairs
    ], ignore_index=True)
    attempts.insert(0, "grid", grid.value)
    attempts["outcome_information_time"] = attempts["resolution_time"]
    attempts["outcome_is_hindsight_at_start"] = attempts[
        "outcome_information_time"
    ].gt(attempts["start_time"])
    attempts_by_grid[grid] = attempts

    resolved = attempts.loc[attempts["outcome"].isin([
        AttemptOutcome.PROPAGATED.value,
        AttemptOutcome.REJECTED.value,
    ])].copy()
    clean = resolved.loc[resolved["resolution_bars"].gt(0)].copy()
    if not clean["outcome_is_hindsight_at_start"].all():
        raise AssertionError("A clean adjacent outcome was already known at initiation.")

    for lower_scale, upper_scale in adjacent_pairs:
        edge = f"{lower_scale.value}->{upper_scale.value}"
        edge_attempts = attempts.loc[attempts["edge"].eq(edge)]
        edge_resolved = resolved.loc[resolved["edge"].eq(edge)]
        edge_clean = clean.loc[clean["edge"].eq(edge)]
        event_count_ratio = len(ledgers[lower_scale]) / len(ledgers[upper_scale])
        edge_records.append({
            "grid": grid.value,
            "edge": edge,
            "lower_threshold": GRID_THRESHOLDS[grid][lower_scale],
            "upper_threshold": GRID_THRESHOLDS[grid][upper_scale],
            "threshold_ratio": GRID_THRESHOLDS[grid][upper_scale] / GRID_THRESHOLDS[grid][lower_scale],
            "all_attempts": len(edge_attempts),
            "same_bar_propagations": edge_resolved["resolution_bars"].eq(0).sum(),
            "clean_future_attempts": len(edge_clean),
            "event_count_ratio": event_count_ratio,
            "approximate_propagation_rate": 2.0 / (event_count_ratio + 1.0),
            "all_resolved_propagation_rate": edge_resolved["outcome"].eq(
                AttemptOutcome.PROPAGATED.value
            ).mean(),
            "clean_propagation_rate": edge_clean["outcome"].eq(
                AttemptOutcome.PROPAGATED.value
            ).mean(),
            "median_clean_resolution_bars": edge_clean["resolution_bars"].median(),
        })

edge_summary = pd.DataFrame.from_records(edge_records).set_index(["grid", "edge"])
display(edge_summary.style.format({
    "lower_threshold": "{:.4%}",
    "upper_threshold": "{:.4%}",
    "threshold_ratio": "{:.3f}",
    "all_attempts": "{:,}",
    "same_bar_propagations": "{:,}",
    "clean_future_attempts": "{:,}",
    "event_count_ratio": "{:.3f}",
    "approximate_propagation_rate": "{:.2%}",
    "all_resolved_propagation_rate": "{:.2%}",
    "clean_propagation_rate": "{:.2%}",
    "median_clean_resolution_bars": "{:.1f}",
}))

In [12]:
propagation_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Clean future propagation rate", "Median clean resolution bars"],
)
for grid in ThresholdGrid:
    summary = edge_summary.loc[grid.value]
    color = "#2563eb" if grid == ThresholdGrid.LINEAR else "#f97316"
    propagation_figure.add_trace(go.Bar(
        x=summary.index,
        y=summary["clean_propagation_rate"],
        name=GRID_LABELS[grid],
        legendgroup=grid.value,
        marker_color=color,
    ), row=1, col=1)
    propagation_figure.add_trace(go.Bar(
        x=summary.index,
        y=summary["median_clean_resolution_bars"],
        name=GRID_LABELS[grid],
        legendgroup=grid.value,
        showlegend=False,
        marker_color=color,
    ), row=1, col=2)

propagation_figure.update_layout(
    title=f"{SYMBOL}: adjacent-scale diagnostic outcomes by threshold grid",
    template="plotly_white",
    height=500,
    barmode="group",
    legend={"orientation": "h", "y": 1.15, "x": 0.0},
)
propagation_figure.update_yaxes(title="Later propagation rate", tickformat=".0%", row=1, col=1)
propagation_figure.update_yaxes(title="Bars", row=1, col=2)
propagation_figure.show()

## Adjacent nesting

Almost every completed upper interval contains an odd number of lower DCCs. One lower event supplies the eventual propagation; every earlier rejected attempt contributes a pair of lower flips. This parity result applies independently to all four adjacent edges. Parity is a near-invariant rather than a law: with multiplicative thresholds the upper clock can, rarely, confirm while the lower clock still points the other way. `build_nesting_intervals` keeps such intervals and flags them with `parity_violation`, so the count below should be read rather than assumed to be zero.

In [13]:
from dc_basics.hierarchy import build_nesting_intervals


nesting_parts = []
for grid in ThresholdGrid:
    ledgers = grid_artifacts[grid]["event_ledgers"]
    for lower_scale, upper_scale in adjacent_pairs:
        intervals = build_nesting_intervals(
            ledgers[lower_scale], ledgers[upper_scale], lower_scale, upper_scale
        )
        intervals.insert(0, "grid", grid.value)
        nesting_parts.append(intervals)

nesting_intervals = pd.concat(nesting_parts, ignore_index=True)
nesting_summary = nesting_intervals.groupby(["grid", "edge"], sort=False).agg(
    upper_intervals=("upper_event_id", "size"),
    parity_violations=("parity_violation", "sum"),
    lower_dcc_count_median=("lower_dcc_count", "median"),
    lower_dcc_count_p90=("lower_dcc_count", lambda values: values.quantile(0.90)),
    prior_rejections_median=("prior_rejections", "median"),
    prior_rejections_p90=("prior_rejections", lambda values: values.quantile(0.90)),
    interval_bars_median=("interval_bars", "median"),
)

display(nesting_summary.style.format({
    "upper_intervals": "{:,}",
    "parity_violations": "{:,}",
    "lower_dcc_count_median": "{:.1f}",
    "lower_dcc_count_p90": "{:.1f}",
    "prior_rejections_median": "{:.1f}",
    "prior_rejections_p90": "{:.1f}",
    "interval_bars_median": "{:.1f}",
}))

## Same price, different causal structural readings

A difference between grids is not an error: the grids ask different excursion-size questions. The comparison below is causal at every retained bar. Only the selection of a particularly persistent divergence interval for the chart is hindsight.

In [14]:
def prefix_state_path(state_path: pd.DataFrame, prefix: str) -> pd.DataFrame:
    columns = [
        "time", "configuration", "hierarchy_shape", "direction_boundaries",
        "outer_agreement_breadth", "coherent_frontier",
        *[f"{scale.value}_mode" for scale in SCALE_ORDER],
    ]
    renamed = state_path[columns].set_index("time")
    return renamed.add_prefix(f"{prefix}_")


comparison = prefix_state_path(
    state_paths[ThresholdGrid.LINEAR], ThresholdGrid.LINEAR.value
).join(
    prefix_state_path(
        state_paths[ThresholdGrid.CONSTANT_RATIO], ThresholdGrid.CONSTANT_RATIO.value
    ),
    how="inner",
)
comparison["same_configuration"] = comparison["linear_configuration"].eq(
    comparison["constant_ratio_configuration"]
)
comparison["same_hierarchy_shape"] = comparison["linear_hierarchy_shape"].eq(
    comparison["constant_ratio_hierarchy_shape"]
)
comparison["boundary_difference"] = (
    comparison["linear_direction_boundaries"] - comparison["constant_ratio_direction_boundaries"]
)
comparison["outer_breadth_difference"] = (
    comparison["linear_outer_agreement_breadth"] - comparison["constant_ratio_outer_agreement_breadth"]
)

grid_agreement_records = [{
    "measure": "full_configuration",
    "agreement_share": comparison["same_configuration"].mean(),
}, {
    "measure": "hierarchy_shape",
    "agreement_share": comparison["same_hierarchy_shape"].mean(),
}]
for scale in SCALE_ORDER:
    grid_agreement_records.append({
        "measure": f"{scale.value}_ordinal_mode",
        "agreement_share": comparison[f"linear_{scale.value}_mode"].eq(
            comparison[f"constant_ratio_{scale.value}_mode"]
        ).mean(),
    })

grid_agreement_summary = pd.DataFrame.from_records(grid_agreement_records).set_index("measure")
display(grid_agreement_summary.style.format({"agreement_share": "{:.2%}"}))

comparison_summary = pd.Series({
    "common_initialized_bars": len(comparison),
    "mean_absolute_boundary_difference": comparison["boundary_difference"].abs().mean(),
    "different_shape_share": (~comparison["same_hierarchy_shape"]).mean(),
    "different_configuration_share": (~comparison["same_configuration"]).mean(),
}, name="value")
display(comparison_summary.to_frame().style.format({
    "value": lambda value: f"{value:,.0f}" if value >= 1_000 else f"{value:.3f}"
}))

,agreement_share
measure,
full_configuration,51.64%
hierarchy_shape,66.77%
small_ordinal_mode,100.00%
medium_ordinal_mode,79.30%
large_ordinal_mode,76.75%
macro_ordinal_mode,86.51%
outer_ordinal_mode,99.31%


,value
common_initialized_bars,"258,334"
mean_absolute_boundary_difference,0.235
different_shape_share,0.332
different_configuration_share,0.484


In [15]:
comparison_with_runs = comparison.reset_index()
comparison_with_runs["bar_position"] = prices.index.get_indexer(comparison_with_runs["time"])
comparison_with_runs["shape_differs"] = ~comparison_with_runs["same_hierarchy_shape"]
comparison_with_runs["divergence_run_id"] = comparison_with_runs["shape_differs"].ne(
    comparison_with_runs["shape_differs"].shift()
).cumsum()

divergence_runs = (
    comparison_with_runs.loc[comparison_with_runs["shape_differs"]]
    .groupby("divergence_run_id", sort=True)
    .agg(
        start_time=("time", "first"),
        end_time=("time", "last"),
        start_position=("bar_position", "first"),
        end_position=("bar_position", "last"),
        bars=("time", "size"),
        linear_shape=("linear_hierarchy_shape", "first"),
        constant_ratio_shape=("constant_ratio_hierarchy_shape", "first"),
    )
    .reset_index(drop=True)
)

recent_cutoff = prices.index[int(len(prices) * 0.75)]
recent_divergences = divergence_runs.loc[divergence_runs["end_time"].ge(recent_cutoff)]
candidate_divergences = recent_divergences if not recent_divergences.empty else divergence_runs
divergence_example = candidate_divergences.sort_values(["bars", "end_time"]).tail(1).iloc[0]
display(divergence_example.to_frame(name="selected_hindsight_example"))


def plot_grid_divergence(
    close: pd.Series, frame: pd.DataFrame, example: pd.Series
) -> go.Figure:
    start_position = max(0, int(example["start_position"]) - DIVERGENCE_BARS_BEFORE)
    end_position = min(len(close), int(example["end_position"]) + DIVERGENCE_BARS_AFTER + 1)
    price_window = close.iloc[start_position:end_position]
    structure_window = frame.loc[price_window.index]
    ordered_scales = list(reversed(SCALE_ORDER))

    figure = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.64, 0.18, 0.18],
        vertical_spacing=0.05,
        subplot_titles=["Close", "Linear grid modes", "Constant-ratio grid modes"],
    )
    figure.add_trace(go.Scattergl(
        x=price_window.index,
        y=price_window.to_numpy(),
        mode="lines",
        name="Close",
        line={"color": "#64748b", "width": 1.2},
        hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
    ), row=1, col=1)
    figure.add_vrect(
        x0=example["start_time"],
        x1=example["end_time"],
        fillcolor="#facc15",
        opacity=0.18,
        line_width=0,
        row=1,
        col=1,
    )

    for row_number, grid in enumerate(ThresholdGrid, start=2):
        columns = [f"{grid.value}_{scale.value}_mode" for scale in ordered_scales]
        mode_matrix = structure_window[columns].apply(
            lambda column: column.map({DcMode.DOWN.value: 0.0, DcMode.UP.value: 1.0})
        ).to_numpy(dtype=np.float64).T
        figure.add_trace(go.Heatmap(
            x=structure_window.index,
            y=[scale.value for scale in ordered_scales],
            z=mode_matrix,
            zmin=0.0,
            zmax=1.0,
            colorscale=[[0.0, "#dc2626"], [0.499, "#dc2626"], [0.5, "#16a34a"], [1.0, "#16a34a"]],
            showscale=row_number == 2,
            colorbar={"title": "Mode", "tickvals": [0.0, 1.0], "ticktext": ["down", "up"]},
            hovertemplate="time=%{x}<br>scale=%{y}<br>mode_code=%{z}<extra></extra>",
        ), row=row_number, col=1)

    figure.update_layout(
        title=(
            f"{SYMBOL}: hindsight-selected grid-divergence case — "
            f"{example['linear_shape']} versus {example['constant_ratio_shape']}"
        ),
        template="plotly_white",
        height=850,
        hovermode="x unified",
        margin={"l": 85, "r": 100, "t": 100, "b": 60},
    )
    figure.update_yaxes(title="Close", row=1, col=1)
    figure.update_xaxes(title="Time", row=3, col=1)
    return figure


plot_grid_divergence(prices, comparison, divergence_example).show()

,selected_hindsight_example
start_time,2026-08-21 19:25:00+02:00
end_time,2026-08-26 15:10:00+02:00
start_position,242495
end_position,243307
bars,813
linear_shape,counter_depth_1
constant_ratio_shape,counter_depth_2


# Five-threshold and grid-design completion gate

Before moving to five-scale linkage, be able to explain:

1. Why multithreshold DC is real while five thresholds is not a canonical calibration.
2. Why five clocks create 32 possible configurations.
3. How to read every configuration in small / medium / large / macro / outer order.
4. Why one boundary is a coherent propagation frontier and two or more boundaries indicate fragmentation.
5. Why agreement breadth does not uniquely identify structural shape.
6. Why the two grids produce the same 0.20% clock but different higher clocks.
7. Why linear spacing has ratios 2.0, 1.5, 1.33, and 1.25.
8. Why the constant-ratio grid holds every adjacent ratio at 1.5.
9. How threshold ratios influence event counts, nesting, and adjacent propagation rates.
10. Why a difference between the two causal structural readings is not a contradiction.
11. Why same-bar multi-scale transitions have unknowable intrabar order on close-only bars.
12. Why completed TMV, T, and R are known only at their DCC confirmation.
13. Why propagated/rejected outcomes remain hindsight until their resolution DCC.
14. Why the selected divergence chart is diagnostic even though both displayed state paths are causal.
15. Why neither grid may be called superior without a predeclared objective and out-of-sample validation.

The next step is five-scale linkage on the grid fixed by the predeclared geometric rule in the interpretation lock below: a constant adjacent ratio. Clock adequacy is a minimum condition checked for both grids, not a selection criterion, and future trading performance plays no part.

# Interpretation lock for the five-scale sprint

The constant-ratio grid is the primary hierarchy for linkage and causal live development:

    0.20% / 0.30% / 0.45% / 0.675% / 1.0125%

**Why this grid — geometry only.** The selection rule is structural and was fixed before any outcome was inspected: every hand-off should ask the same relative scale question, so the primary ladder must hold the adjacent threshold ratio constant. The constant-ratio grid does so by construction (every ratio is 1.5); the linear grid does not (2.0, 1.5, 1.33, 1.25). Clock adequacy is a minimum condition, not a selection criterion, and it does not separate the grids here: both produced all 32 configurations and both outer clocks retained 146 completed events.

The linear grid remains a sensitivity benchmark.

**Post-hoc observations — recorded after the lock, not reasons for it.** The constant-ratio grid's clean adjacent propagation rates were 66.75%, 64.63%, 61.52%, and 64.13%; the linear grid's rose from 46.11% to 79.44%. Both patterns are largely arithmetic (see the edge table): an adjacent rate is close to $2/(N_{lower}/N_{upper}+1)$, and event-count ratios follow threshold ratios. Equal ratios therefore give similar rates and shrinking ratios give rising rates. Neither pattern is evidence that upper-scale propagation is economically easier or harder, and neither may be used to choose a grid (Chapter 18, exercise 5).

The grids agreed on the complete configuration for 51.62% of common initialized bars and on hierarchy shape for 66.78%. Therefore a multiscale state is conditional on the declared threshold geometry. The grid is part of the model specification, not a neutral preprocessing detail.

The primary grid is now frozen for the linkage and live-development notebooks. Robustness will separately test symbols, absolute scale, calendar periods, direction, and the linear-grid sensitivity case.